# Cleaning — `postings.csv` and `companies.csv`

This notebook resolves the open questions flagged in `02_eda_postings.ipynb` and `03_eda_companies_and_jobs.ipynb`. Every decision below states the evidence that grounded it before making the change — nothing here is arbitrary.

**Decision log:**

| Question | Decision | Why |
|---|---|---|
| `closed_time` (99.1% missing) | Drop the column | Too sparse to be usable; not needed for recommendation logic |
| Timestamp columns (epoch ms) | Convert to datetime | Needed for any future time-based analysis |
| `remote_allowed` semantics | Keep original column; add derived `is_remote` (missing -> `False`) | Column only ever contains `1.0` or missing (no explicit "not remote" value exists) — missing is the only way "not remote" is represented in this data |
| Salary outliers | Flag rows outside a defensible USD range as unreliable, don't drop postings | See evidence below — only ~1.3% of USD salary rows are affected |
| `title`/`company_name` text | Strip whitespace, collapse internal spaces | Minimal, safe normalization; full title-taxonomy standardization deferred to the skill-extraction stage |
| `companies.csv` placeholder values | Replace literal `"0"`/`"-"` with real `NaN` in location fields | Confirmed present in `country`, `state`, `city`, `zip_code`, `address` — see evidence below |

Not addressed here (deferred, documented as future work): `med_salary` (94.9% missing, redundant with min/max), `benefits.csv`'s `inferred` flag (kept as-is, just documented), `company_specialities.csv` coverage gap (kept as-is).

## Setup

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

BASE = "./"
postings = pd.read_csv(f"{BASE}postings.csv")
companies = pd.read_csv(f"{BASE}companies/companies.csv")
print(f"postings: {postings.shape}, companies: {companies.shape}")

postings: (123849, 31), companies: (24473, 10)


## 1. Drop `closed_time`

99.1% missing (confirmed in the EDA notebook) — not usable as a feature, and not needed for the recommendation pipeline.

In [2]:
postings = postings.drop(columns=["closed_time"])
print("dropped closed_time; columns remaining:", postings.shape[1])

dropped closed_time; columns remaining: 30


## 2. Convert timestamp columns

`listed_time`, `original_listed_time`, `expiry` are epoch milliseconds. Converting to real datetimes.

In [3]:
for col in ["listed_time", "original_listed_time", "expiry"]:
    postings[col] = pd.to_datetime(postings[col], unit="ms")

postings[["listed_time", "original_listed_time", "expiry"]].agg(["min", "max"])

,listed_time,original_listed_time,expiry
min,2024-03-24 21:50:14,2023-12-05 21:08:53,2024-04-12 06:30:48
max,2024-04-20 00:26:56,2024-04-20 00:26:43,2024-10-17 00:26:36


## 3. `remote_allowed` -> derived `is_remote`

**Evidence:** in the raw data, `remote_allowed` only ever takes the value `1.0` or is missing — there is no `0.0` anywhere. That means missing is not ambiguous here; it's the only way "not marked remote" is represented. We keep the original column (in case that assumption turns out wrong later) and add a clean boolean.

In [4]:
assert set(postings["remote_allowed"].dropna().unique()) == {1.0}, "remote_allowed has values other than 1.0 - assumption broken, revisit"

postings["is_remote"] = postings["remote_allowed"].fillna(0).astype(bool)
print(postings["is_remote"].value_counts())

is_remote
False    108603
True      15246
Name: count, dtype: int64


## 4. Salary outlier flag

**Evidence:** restricting to `currency == "USD"` (99.96% of rows with a currency at all), the percentile distribution jumps sharply at the extremes:

| Percentile | normalized_salary |
|---|---|
| p0.5 | \$36 |
| p1 | \$130 |
| p50 | \$81,500 |
| p99 | \$300,973 |
| p100 | \$535,600,000 |

Using \$10,000/year as a floor (below full-time minimum wage) and \$1,000,000/year as a ceiling (comfortably above legitimate senior executive comp), only **454 of 36,058 USD rows (1.3%)** fall outside that range — mostly YEARLY-labeled rows that look like mislabeled hourly rates, plus a handful of clearly erroneous multi-million-dollar entries. We flag these as unreliable rather than dropping the postings entirely, since the rest of the posting (title, description, skills) is still usable.

In [5]:
SALARY_FLOOR = 10_000
SALARY_CEILING = 1_000_000

postings["salary_reliable"] = (
    (postings["currency"] == "USD")
    & postings["normalized_salary"].between(SALARY_FLOOR, SALARY_CEILING)
)

has_salary = postings["normalized_salary"].notna()
print(f"postings with any normalized_salary: {has_salary.sum():,}")
print(f"flagged reliable: {postings['salary_reliable'].sum():,}")
print(f"has salary but flagged unreliable: {(has_salary & ~postings['salary_reliable']).sum():,}")

postings with any normalized_salary: 36,073


flagged reliable: 35,604
has salary but flagged unreliable: 469


## 5. Minimal text normalization

Just whitespace cleanup on `title` and `company_name` — safe and unambiguous. Full standardization (e.g. mapping "Sr. Software Engineer" and "Senior Software Engineer" together) is deferred to the skill-extraction stage, where we'll need NLP tooling anyway.

In [6]:
for col in ["title", "company_name"]:
    postings[col] = postings[col].str.strip().str.replace(r"\s+", " ", regex=True)

postings[["title", "company_name"]].head()

,title,company_name
0,Marketing Coordinator,Corcoran Sawyer Smith
1,Mental Health Therapist/Counselor,NaN
2,Assitant Restaurant Manager,The National Exemplar
3,Senior Elder Law / Trusts and Estates Associat...,"Abrams Fensterman, LLP"
4,Service Technician,NaN


## 6. `companies.csv` placeholder values

**Evidence:** literal `"0"` and `"-"` strings appear in place of real nulls:

| Column | `"0"` count | `"-"` count |
|---|---|---|
| country | 727 | 0 |
| state | 2,175 | 3 |
| city | 1,004 | 1 |
| zip_code | 3,060 | 0 |
| address | 3,972 | 6 |

Replacing these with real `NaN` so missingness checks and any downstream groupby are accurate.

In [7]:
location_cols = ["country", "state", "city", "zip_code", "address"]
before_missing = companies[location_cols].isna().sum()

companies[location_cols] = companies[location_cols].astype(str).replace({"0": np.nan, "-": np.nan, "nan": np.nan})

after_missing = companies[location_cols].isna().sum()
pd.DataFrame({"missing_before": before_missing, "missing_after": after_missing})

,missing_before,missing_after
country,0,727
state,22,2200
city,1,1006
zip_code,28,3088
address,22,4000


## 7. Save cleaned tables

Saved as Parquet (preserves dtypes, including the new datetime columns) alongside the raw CSVs. These are gitignored like the raw data — regenerate by running this notebook.

In [8]:
postings.to_parquet(f"{BASE}postings_clean.parquet", index=False)
companies.to_parquet(f"{BASE}companies_clean.parquet", index=False)
print("saved postings_clean.parquet and companies_clean.parquet")
print(f"final postings shape: {postings.shape}")
print(f"final companies shape: {companies.shape}")

saved postings_clean.parquet and companies_clean.parquet
final postings shape: (123849, 32)
final companies shape: (24473, 10)
